# 05b City and relief

Phase 5 build check: the new design utilities at city scale (roads, buildings) and three relief designs. Run `00b_city_data.ipynb` first.

Record pass or fail in the table at the end, with screenshots of anything that looks off.

In [14]:
import warnings
import webbrowser
from pathlib import Path

import geopandas as gpd
import ipywidgets as widgets
from IPython.display import display
from lonboard import Map

import kharita as kh
from kharita import basemaps as kb

warnings.filterwarnings("ignore", module="lonboard")
states = gpd.read_file("../data/processed/sudan_states.gpkg")
roads = gpd.read_file("../data/processed/khartoum_roads.gpkg")
buildings = gpd.read_file("../data/processed/khartoum_buildings.gpkg")
major = roads[roads["rank"] >= 4]
minor = roads[roads["rank"].between(1, 3)]
KHARTOUM = {"longitude": 32.53, "latitude": 15.60, "zoom": 14}
OSM = "Roads and buildings: Overture Maps Foundation, OpenStreetMap contributors (ODbL)."
len(roads), len(buildings)

(10706, 71223)

In [15]:
# A. Every theme at city zoom: basemap roads, buildings and street names, no data
for name in kb.THEMES:
    display(widgets.HTML(f"<b>{name}</b>"))
    display(Map([], basemap=kb.basemap(name), view_state=KHARTOUM, height=300))

HTML(value='<b>paper</b>')

HTML(value='<b>midnight</b>')

HTML(value='<b>mono</b>')

HTML(value='<b>neon</b>')

HTML(value='<b>muted</b>')

In [16]:
# B. Basemap buildings in 3D, from the vector tiles alone
tilted = {**KHARTOUM, "zoom": 15, "pitch": 55, "bearing": -20}
Map([], basemap=kb.basemap("midnight", buildings="3d"), view_state=tilted, height=420)

In [17]:
# C. Roads coloured by class, street names drawn on top
kh.map(
    roads,
    "theme-midnight stroke-[road_class] ramp-set2 stroke-1.5 tooltip-[name,road_class]",
    legend={"title": "Road class", "note": OSM},
)

In [18]:
# D. Traffic look: faint minor streets, major roads with halo and glow on neon
m = kh.Map(height=520)
m.add(minor, "stroke-0.8 stroke-violet-400/40 basemap-neon buildings-none labels-none legend-none")
m.add(major, "stroke-2.5 stroke-[road_class] ramp-plasma halo-fuchsia-400/70 glow", legend={"title": "Road class", "note": OSM})
m

In [19]:
# E. Building footprints by size on paper
kh.map(
    buildings,
    "theme-paper fill-[area_m2] scheme-q5 stroke-0.3 stroke-white/60 tooltip-[kind,area_m2]",
    legend={"title": "Footprint, m2", "fmt": "{:,.0f}", "note": OSM},
)

In [20]:
# F. Buildings raised to their height in metres; extrude-max equal to the tallest keeps true scale
tallest = buildings["height_m"].max()
kh.map(
    buildings,
    f"theme-midnight fill-[height_m] scheme-[7,12,20,40] extrude-[height_m] extrude-max-{tallest:g} "
    "buildings-none tooltip-[kind,height_m,height_source]",
    legend={"title": "Height, m", "note": "Unmapped heights assumed 6 m. " + OSM},
)

In [21]:
# G. Buildings by type on mono
kh.map(buildings, "theme-mono fill-[kind] ramp-set2 stroke-0.3 stroke-white", legend={"title": "Building type", "note": OSM})

C:\Users\dev\Desktop\Projects\kharita\kharita\binder.py:169: KharitaWarning: kind has 24 categories; the 17 least common are grouped as 'other'.
  fill, fill_scale = (_colour(spec, data, spec.fill_by) if spec.fill_by is not None


In [22]:
# H. Three relief designs on paper at Jebel Marra: soft, sharp, tint
JEBEL_MARRA = {"longitude": 24.25, "latitude": 12.95, "zoom": 8}
for relief in kb.RELIEFS:
    display(widgets.HTML(f"<b>relief-{relief}</b>"))
    display(Map(kh.layer(states, "fill-transparent stroke-1 stroke-slate-700"), basemap=kb.basemap("paper", relief=relief), view_state=JEBEL_MARRA, height=320))

HTML(value='<b>relief-soft</b>')

HTML(value='<b>relief-sharp</b>')

HTML(value='<b>relief-tint</b>')

In [23]:
# I. Relief on dark themes, now tuned to show: midnight sharp, neon tint
RED_SEA_HILLS = {"longitude": 36.6, "latitude": 19.2, "zoom": 7}
for theme, relief in (("midnight", "sharp"), ("neon", "tint")):
    display(widgets.HTML(f"<b>{theme}, relief-{relief}</b>"))
    display(Map(kh.layer(states, "fill-transparent stroke-1 stroke-slate-400"), basemap=kb.basemap(theme, relief=relief), view_state=RED_SEA_HILLS, height=320))

HTML(value='<b>midnight, relief-sharp</b>')

HTML(value='<b>neon, relief-tint</b>')

In [24]:
# J. Whole country on basemap-terrain, outlined with a halo, all from the style string
kh.map(states, "preset-outline stroke-slate-600 basemap-terrain halo-white")

In [27]:
# K. Choropleth on midnight: halo, labels above the colours, theme ramp
kh.map(
    states,
    "theme-midnight fill-[density] scheme-q5 stroke-1 stroke-cyan-200 halo-cyan-300 opacity-90 tooltip-[key,density]",
    legend={"title": "People per km2"},
)

In [26]:
# L. Export the traffic map from D; glow and the neon theme travel with it
out = Path("../spike_out")
out.mkdir(exist_ok=True)
m.to_html(out / "khartoum_roads.html", title="Khartoum roads")
webbrowser.open((out / "khartoum_roads.html").resolve().as_uri())

True

## Results

| Test | What passing looks like | Pass? | Notes |
| --- | --- | --- | --- |
| A | Five themes show streets, footprints and street names at city zoom | yes | map views are normal midnight mono neon muted the best one for me is the muted but this debend on the user that create the map they want |
| B | Buildings stand up in 3D, darker navy on midnight | yes | works good question with out creating a new map is it possible to have different thems for 3d or is it only midnight |
| C | Roads in eight colours, street names readable over them, hover shows name and class | yes | questions can the user change the colors for the road or is it fixed im thinking do we have a color collation for example he say light it have a list of 10 colors for the roads or is this fixed let me know as i want more options for th eusers with coustimizaitons |
| D | Major roads glow, brighter where they cross; minor streets faint | yes | same question as c and also can 3d map have road map like this or is it not possible |
| E | Footprints in five yellow-red classes, labels on top | yes | it works but same questions as c and d |
| F | Buildings raised; tall ones in the top classes; tilted view | yes | same as the above |
| G | Building types in distinct colours, rare types grouped as other | yes | can i user control the colors usinging already set thems for the colors of the building type not the map theme the same as tailwindcss |
| H | Three clearly different relief looks; tint shows colour by height | yes | im thinking we can add a new design the topographic map and elevation maps degin that show the hight in form of steps top view |
| I | Relief now visible on both dark themes | yes | neo relief tint looks better |
| J | Tinted relief across Sudan, white halo around states | yes | looks great |
| K | Viridis classes, cyan halo, state names above the colours | yes | look great |
| L | Exported page matches D | yes | its working |

Which relief design do you like best for each theme?